# 08 - End-to-End Remote Ollama AI Scoring Engine Test

This notebook allows you to configure your remote Ollama server settings (URL, model name, etc.) and run automated AI scoring across Marine CSMS submissions with comprehensive Excel export.

In [ ]:
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath('..'))

# Configure remote Ollama Server settings dynamically based on your environment
config_path = os.path.abspath('../configs/model_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    config['active_provider'] = 'ollama'
    
    # CUSTOMIZE YOUR OLLAMA SETTINGS BELOW:
    config['providers']['ollama']['base_url'] = 'http://localhost:5162/'
    config['providers']['ollama']['model'] = 'qwen3.5:27b'
    config['providers']['ollama']['temperature'] = 0.0
    
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=4)
    print(f'[CONFIG] Active provider set to: {config["active_provider"]}')
    print(f'[CONFIG] Ollama Server URL: {config["providers"]["ollama"]["base_url"]}')
    print(f'[CONFIG] Ollama Model: {config["providers"]["ollama"]["model"]}')

from engine.pipeline import evaluate_all_categories, discover_category_folders, score_category

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

## Step 1 & 2: Discover Category Folders

In [ ]:
marine_dir = os.path.abspath('../test_submissions/marine')
discovered = discover_category_folders(marine_dir)
print(f'Discovered {len(discovered)} category folders in {marine_dir}:')
for d in discovered:
    print(f' - Category {d["category_id"]}: {d["folder_name"]} with path {d["path"]}')

## Step 3: Test Category 3.5 End-to-End via Ollama Provider

In [ ]:
# Test Category 3.5 specifically using Ollama model
cat_35_folders = [d for d in discovered if d['category_id'] == '3.5']
print(f'Found {len(cat_35_folders)} folder(s) for Category 3.5:')
for d in cat_35_folders:
    print(f'\n========================================================')
    print(f'Testing Category 3.5 (Ollama): {d["folder_name"]}')
    print(f'========================================================')
    res = score_category(d['category_id'], d['path'], silent=False)
    print('\nResult Summary:')
    print(json.dumps({
        'category_id': res.get('category_id'),
        'folder_name': res.get('folder_name'),
        'expected_score': res.get('expected_score'),
        'predicted_score': res.get('predicted_score'),
        'confidence': res.get('confidence'),
        'status': res.get('status'),
        'evidence_count': res.get('evidence_count'),
        'processing_time_sec': res.get('processing_time_sec')
    }, indent=2))

## Step 4: Evaluate All Categories via Ollama Provider

In [ ]:
# This runs the pipeline for all categories using Ollama, logs results, and automatically writes JSON, CSV, Excel (.xlsx), and HTML Viewer
report = evaluate_all_categories(marine_dir)

print('\nSummary KPI Metrics:')
print(f'Total Categories : {report.get("total_categories", 0)}')
print(f'Passed (Correct) : {report.get("passed", 0)}')
print(f'Needs Review     : {report.get("needs_review", 0)}')
print(f'Failed / Mismatch: {report.get("failed", 0)}')
print(f'Accuracy         : {report.get("accuracy", 0.0)}%')

## Step 5: View Results Table & Comprehensive Report Links

In [ ]:
if report.get('results'):
    df = pd.DataFrame(report['results'])
    display(df[['category_id', 'folder_name', 'expected_score', 'predicted_score', 'confidence', 'status', 'evidence_count', 'processing_time_sec']])

excel_path = os.path.abspath('../outputs/reports/evaluation_report.xlsx')
csv_path = os.path.abspath('../outputs/reports/evaluation_report.csv')
viewer_path = os.path.abspath('../viewer/index.html')

print(f'\n[EXCEL REPORT] Comprehensive report exported to:\nfile:///{excel_path.replace(os.sep, "/")}')
print(f'\n[CSV REPORT] CSV report exported to:\nfile:///{csv_path.replace(os.sep, "/")}')
print(f'\n[VIEWER] Interactive HTML Viewer is ready at:\nfile:///{viewer_path.replace(os.sep, "/")}')
print('You can open the HTML file directly in any browser or open the Excel report for offline analysis!')

# Restore default provider to openai after test
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    config['active_provider'] = 'openai'
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=4)
    print('[CONFIG] Restored default provider to: openai')